## Loading MET_nowcast dataset
This dataset contains hourly meteorological nowcast observations used for wind power forecasting. Each record is indexed by time and wind park location and includes key weather variables such as air temperature at 2 m, sea-level air pressure, relative humidity, precipitation amount, wind speed at 10 m, and wind direction at 10 m.

In [113]:
import pandas as pd

met_nowcast = pd.read_parquet("../src/raw_data/met_nowcast.parquet")
met_nowcast.head()

,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,precipitation_amount,wind_speed_10m,wind_direction_10m,windpark
time,,,,,,,
2020-01-01 00:00:00,276.363831,100703.859375,0.966131,3.147963,13.445930,250.681976,Bessakerfjellet
2020-01-01 01:00:00,276.951080,100595.609375,0.972677,3.636308,13.807796,251.660736,Bessakerfjellet
2020-01-01 02:00:00,277.113007,100526.656250,0.964682,1.921109,15.788631,250.626617,Bessakerfjellet
2020-01-01 03:00:00,277.259033,100408.359375,0.966498,1.919362,16.493980,254.728180,Bessakerfjellet
2020-01-01 04:00:00,277.229248,100421.062500,0.964835,1.275917,16.933884,253.571869,Bessakerfjellet


## Loading met_forecast dataset

This dataset contains forecasted meteorological variables used as input features for wind power prediction. Each row corresponds to a specific time window for a wind park (or substation), and columns include multiple forecast horizons and weather variables (e.g., wind speed components, wind direction, temperature/pressure-related features). The dataset is typically used to build feature matrices for forecasting future wind power at either wind park or bidding-zone level.

In [114]:
met_forecast = pd.read_parquet("../src/raw_data/met_forecast.parquet")

In [115]:
print(f"The met_forecast has the following: {met_forecast.shape[0]} rows, {met_forecast.shape[1]} columns and {met_forecast.isnull().sum()} missing values")

The met_forecast has the following: 6874212 rows, 94 columns and sid         0
time_ref    0
time        0
lt          0
ws10m_00    0
           ..
g10m_10     0
g10m_11     0
g10m_12     0
g10m_13     0
g10m_14     0
Length: 94, dtype: int64 missing values


Here we are identifying the ensemble columns

In [116]:
import re

# Find all ensemble feature prefixes (ws10m, g10m, etc.)
prefixes = set(
    re.match(r"(.*)_\d+$", c).group(1)
    for c in met_forecast.columns
    if re.match(r".*_\d+$", c)
)

print(prefixes)

{'ws10m', 'rh2m', 't2m', 'wd10m', 'g10m', 'mslp'}


Here we are calculating the mean and std for each ensemble per row (dimesnioanlity reduction)

In [117]:
for prefix in prefixes:
    cols = [column for column in met_forecast.columns if column.startswith(prefix + "_")]

    met_forecast[f"{prefix}_mean"] = met_forecast[cols].mean(axis=1)
    met_forecast[f"{prefix}_std"] = met_forecast[cols].std(axis=1)

Drop the original Ensemble columns after calculating its mean and std

In [118]:
ensemble_cols = [column for column in met_forecast.columns if re.match(r".*_\d+$", column)]
met_forecast = met_forecast.drop(columns=ensemble_cols)
met_forecast.head()

,sid,time_ref,time,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,wd10m_mean,wd10m_std,g10m_mean,g10m_std,mslp_mean,mslp_std
0,Engerfjellet,2020-02-15 12:00:00,2020-02-15 12:00:00,0,2.139608,0.448171,0.823404,0.051589,276.379494,0.395486,197.466405,11.766175,6.050247,1.032772,100502.532600,38.879500
1,Engerfjellet,2020-02-15 12:00:00,2020-02-15 13:00:00,1,3.384011,0.476801,0.825629,0.047831,276.422947,0.375587,207.005511,6.122282,8.121981,0.954943,100555.149267,21.675226
2,Engerfjellet,2020-02-15 12:00:00,2020-02-15 14:00:00,2,3.116065,0.380970,0.847620,0.059136,276.095935,0.374416,197.767234,7.315636,8.276285,0.832413,100519.478200,44.229292
3,Engerfjellet,2020-02-15 12:00:00,2020-02-15 15:00:00,3,2.856627,0.388535,0.886108,0.062451,275.603674,0.339881,186.196795,7.583082,7.532237,0.792371,100468.608667,49.885909
4,Engerfjellet,2020-02-15 12:00:00,2020-02-15 16:00:00,4,3.303811,0.599790,0.942954,0.047610,274.942599,0.471217,185.264509,4.739498,7.929002,1.163573,100395.238133,66.165161


In [119]:
met_forecast_describe = met_forecast.drop(columns=["time_ref", "time", "lt"])
met_forecast_describe.describe()

,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,wd10m_mean,wd10m_std,g10m_mean,g10m_std,mslp_mean,mslp_std
count,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06,6.874212e+06
mean,5.045969e+00,1.027904e+00,8.130827e-01,4.988056e-02,2.784087e+02,7.401979e-01,1.879689e+02,3.178895e+01,9.199543e+00,1.650610e+00,1.010250e+05,1.040311e+02
std,3.138610e+00,6.541391e-01,1.359124e-01,3.087793e-02,7.081419e+00,4.874159e-01,8.009613e+01,3.967216e+01,4.943004e+00,1.024363e+00,1.265245e+03,7.694384e+01
min,1.492034e-01,2.620698e-02,1.431430e-01,6.399405e-08,2.478083e+02,2.858019e-02,2.311048e+00,2.447184e-01,8.071314e-01,4.875700e-02,9.422968e+04,7.818721e+00
25%,2.673073e+00,5.714487e-01,7.296681e-01,2.741771e-02,2.736009e+02,4.119716e-01,1.283403e+02,6.868019e+00,5.464873e+00,9.353666e-01,1.002295e+05,5.011817e+01
50%,4.306994e+00,8.612750e-01,8.357129e-01,4.403964e-02,2.781761e+02,6.119909e-01,1.853189e+02,1.387011e+01,8.213005e+00,1.388829e+00,1.011123e+05,8.211374e+01
75%,6.723898e+00,1.301748e+00,9.219823e-01,6.660115e-02,2.836025e+02,9.189658e-01,2.510621e+02,3.706733e+01,1.197365e+01,2.082409e+00,1.018940e+05,1.334685e+02
max,3.103080e+01,9.202564e+00,1.000007e+00,2.882754e-01,3.052230e+02,7.564517e+00,3.583886e+02,1.848669e+02,4.736941e+01,1.574159e+01,1.051422e+05,1.128533e+03


In [120]:
print(f"The met_forecast has the following: {met_forecast.shape[0]} rows, {met_forecast.shape[1]} columns and {met_forecast.isnull().sum()} missing values")

The met_forecast has the following: 6874212 rows, 16 columns and sid           0
time_ref      0
time          0
lt            0
ws10m_mean    0
ws10m_std     0
rh2m_mean     0
rh2m_std      0
t2m_mean      0
t2m_std       0
wd10m_mean    0
wd10m_std     0
g10m_mean     0
g10m_std      0
mslp_mean     0
mslp_std      0
dtype: int64 missing values


In [121]:
met_nowcast_describe = met_nowcast.drop(columns=["windpark"])
met_nowcast_describe.describe()

,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,precipitation_amount,wind_speed_10m,wind_direction_10m
count,2.806427e+06,2.806427e+06,2.806427e+06,2.806427e+06,2.806427e+06,2.806427e+06
mean,2.782917e+02,1.009934e+05,8.200313e-01,1.642456e-01,5.117610e+00,1.891847e+02
std,7.403435e+00,1.284945e+03,1.442504e-01,5.348223e-01,3.358208e+00,9.344193e+01
min,2.403651e+02,9.424848e+04,1.605081e-01,0.000000e+00,1.352044e-02,1.194908e-04
25%,2.732495e+02,1.001911e+05,7.259581e-01,0.000000e+00,2.547191e+00,1.209505e+02
50%,2.781031e+02,1.010935e+05,8.456026e-01,4.135296e-05,4.344584e+00,1.877242e+02
75%,2.835947e+02,1.018805e+05,9.416699e-01,4.185914e-02,6.957935e+00,2.648553e+02
max,3.066199e+02,1.050429e+05,1.000008e+00,2.889541e+01,3.188647e+01,3.599998e+02


In [122]:
print(f"The met_nowcast has the following: {met_nowcast.shape[0]} rows, {met_nowcast.shape[1]} columns and {met_nowcast.isnull().sum()} missing values")

The met_nowcast has the following: 2806427 rows, 7 columns and air_temperature_2m           0
air_pressure_at_sea_level    0
relative_humidity_2m         0
precipitation_amount         0
wind_speed_10m               0
wind_direction_10m           0
windpark                     0
dtype: int64 missing values


In [123]:
met_nowcast_location_list = met_nowcast["windpark"].unique().tolist()
print(f"The met_nowcast dataset has the following windparks: {met_nowcast_location_list}")

The met_nowcast dataset has the following windparks: ['Bessakerfjellet', 'Bjerk_VK Vindpark', 'Buheii Vindpark', 'Dønnesfjord Vind', 'Egersund Vindkrv', 'Einarsdalen', 'Engerfjellet', 'Fakken', 'Frøya Vindpark', 'Geitfjellet', 'Gismarvik Vindpark', 'Guleslettene Vindpark', 'Hamnefjell', 'Haraheia', 'Haram Kraft', 'Harbaksfjellet', 'Havøygavlen', 'Hennøy', 'Hitra', 'Hundhammerfjelle', 'Hån Vindpark', 'Høg Jæren', 'Kjølberget', 'Kjøllefjord vindpark', 'Kvenndalsfjellet', 'Kvitfjell vindpark', 'Lista VK', 'Lutelandet', 'Marker Vindpark', 'Mehuken', 'Midtfjellet', 'Måkaknuten', 'Nygårdsfjellet', 'Okla Vindkraftverk', 'Raggovidda', 'Raskiftet', 'Raudfjell Vindpark', 'Sandøy Vindkraft', 'Skomakerfjellet', 'Skudeneshavn', 'Smøla', 'Songkjølen', 'Stokkeland', 'Stokkfjellet', 'Storheia', 'Storøy vindpark', 'Svåheia', 'Sørfj', 'Sørmarkfjellet', 'Tellenes', 'Tindafjellet', 'Tonst', 'Tysvær Vindpark', 'Valsneset', 'Valsn_Vimle', 'Vardafjell', 'Ytre Vikna', 'Ånstadblåheia', 'Øie', 'Øyfjell1', 'Øyfj

In [124]:
met_forecast_location_list = met_forecast["sid"].unique().tolist()
print(f"The met_nowcast dataset has the following windparks: {met_forecast_location_list}")

The met_nowcast dataset has the following windparks: ['Engerfjellet', 'Marker Vindpark', 'Kjølberget', 'Songkjølen', 'Hån Vindpark', 'Raskiftet', 'Stokkeland', 'Måkaknuten', 'Buheii Vindpark', 'Vardafjell', 'Bjerk_VK Vindpark', 'Tonst. Vindpark', 'Midtfjellet', 'Tindafjellet', 'Storøy vindpark', 'Gismarvik Vindpark', 'Øie', 'Skudeneshavn', 'Tysvær Vindpark', 'Lista VK', 'Svåheia', 'Egersund Vindkrv', 'Tellenes', 'Høg Jæren', 'Smøla', 'Sørmarkfjellet', 'Ytre Vikna', 'Geitfjellet', 'Storheia', 'Hundhammerfjelle', 'Valsneset', 'Haraheia', 'Okla Vindkraftverk', 'Stokkfjellet', 'Hitra', 'Bessakerfjellet', 'Frøya Vindpark', 'Einarsdalen', 'Haram Kraft', 'Mehuken', 'Lutelandet', 'Hennøy', 'Kvenndalsfjellet', 'Harbaksfjellet', 'Guleslettene Vindpark', 'Skomakerfjellet', 'Sandøy Vindkraft', 'Valsn_Vimle', 'Øyfjell1', 'Fakken', 'Øyfjell2', 'Nygårdsfjellet', 'Raggovidda', 'Kjøllefjord vindpark', 'Sørfj. Vindkraft', 'Dønnesfjord Vind', 'Kvitfjell vindpark', 'Ånstadblåheia', 'Havøygavlen', 'Raudfje

Here we are modifying the mismatch in the names

In [125]:

def clean_name(name):
    name = name.strip()
    name = name.replace(". Vindpark", "")
    name = name.replace(". Vindkraft", "")
    name = name.replace(" Vindkraft", "")
    return name

met_nowcast["windpark_clean"] = met_nowcast["windpark"].apply(clean_name)
met_forecast["sid_clean"] = met_forecast["sid"].apply(clean_name)

print(f"The mismatched windparks between forecast and nowcast dataset are in the following set {set(met_nowcast["windpark_clean"]) - set(met_forecast["sid_clean"])}")


The mismatched windparks between forecast and nowcast dataset are in the following set set()


In [126]:
met_nowcast = met_nowcast.drop(columns=["windpark"])
met_forecast = met_forecast.drop(columns=["sid"])

In [127]:
print("Nowcast time range:")
print(met_nowcast.index.min(), "until", met_nowcast.index.max())

print("\nForecast issue time range:")
print(met_forecast["time_ref"].min(), "until", met_forecast["time_ref"].max())

Nowcast time range:
2020-01-01 00:00:00 until 2025-03-31 23:00:00

Forecast issue time range:
2020-02-15 12:00:00 until 2025-03-24 09:00:00


In [128]:
now_min, now_max = met_nowcast.index.min(), met_nowcast.index.max()
fc_min, fc_max   = met_forecast["time_ref"].min(), met_forecast["time_ref"].max()

start = max(now_min, fc_min)
end   = min(now_max, fc_max)

print("Overlap window:", start, "until", end)

Overlap window: 2020-02-15 12:00:00 until 2025-03-24 09:00:00


In [129]:
met_nowcast = met_nowcast[
    (met_nowcast.index >= start) & (met_nowcast.index <= end)
].copy()
met_forecast = met_forecast[
    met_forecast["time_ref"].between(start, end)
].copy()

In [130]:
met_nowcast.index = pd.to_datetime(met_nowcast.index)
met_forecast["time"] = pd.to_datetime(met_forecast["time"])
met_forecast["time_ref"] = pd.to_datetime(met_forecast["time_ref"])

In [131]:
met_forecast.head()

,time_ref,time,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,wd10m_mean,wd10m_std,g10m_mean,g10m_std,mslp_mean,mslp_std,sid_clean
0,2020-02-15 12:00:00,2020-02-15 12:00:00,0,2.139608,0.448171,0.823404,0.051589,276.379494,0.395486,197.466405,11.766175,6.050247,1.032772,100502.532600,38.879500,Engerfjellet
1,2020-02-15 12:00:00,2020-02-15 13:00:00,1,3.384011,0.476801,0.825629,0.047831,276.422947,0.375587,207.005511,6.122282,8.121981,0.954943,100555.149267,21.675226,Engerfjellet
2,2020-02-15 12:00:00,2020-02-15 14:00:00,2,3.116065,0.380970,0.847620,0.059136,276.095935,0.374416,197.767234,7.315636,8.276285,0.832413,100519.478200,44.229292,Engerfjellet
3,2020-02-15 12:00:00,2020-02-15 15:00:00,3,2.856627,0.388535,0.886108,0.062451,275.603674,0.339881,186.196795,7.583082,7.532237,0.792371,100468.608667,49.885909,Engerfjellet
4,2020-02-15 12:00:00,2020-02-15 16:00:00,4,3.303811,0.599790,0.942954,0.047610,274.942599,0.471217,185.264509,4.739498,7.929002,1.163573,100395.238133,66.165161,Engerfjellet


In [132]:
met_nowcast = met_nowcast.reset_index()              
met_nowcast = met_nowcast.rename(columns={"windpark_clean": "sid_clean"})
met_nowcast.head()

,time,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,precipitation_amount,wind_speed_10m,wind_direction_10m,sid_clean
0,2020-02-15 12:00:00,276.335876,99795.664062,0.821845,2.238679e-04,8.104369,246.912506,Bessakerfjellet
1,2020-02-15 13:00:00,276.076935,99832.359375,0.891700,0.000000e+00,6.657547,233.088379,Bessakerfjellet
2,2020-02-15 14:00:00,275.843689,99830.406250,0.881385,9.209055e-05,5.876413,229.860733,Bessakerfjellet
3,2020-02-15 15:00:00,274.957886,99853.375000,0.837362,0.000000e+00,3.329724,203.135254,Bessakerfjellet
4,2020-02-15 16:00:00,273.522888,99767.031250,0.812765,8.627482e-15,3.800424,142.939713,Bessakerfjellet


In [133]:
merged = met_forecast.merge(
    met_nowcast,
    left_on=["sid_clean", "time_ref"],
    right_on=["sid_clean", "time"],
    how="left",
    suffixes=("", "_now")
).drop(columns=["time_now"])


In [134]:
merged.head()

,time_ref,time,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,wd10m_mean,...,g10m_std,mslp_mean,mslp_std,sid_clean,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,precipitation_amount,wind_speed_10m,wind_direction_10m
0,2020-02-15 12:00:00,2020-02-15 12:00:00,0,2.139608,0.448171,0.823404,0.051589,276.379494,0.395486,197.466405,...,1.032772,100502.532600,38.879500,Engerfjellet,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
1,2020-02-15 12:00:00,2020-02-15 13:00:00,1,3.384011,0.476801,0.825629,0.047831,276.422947,0.375587,207.005511,...,0.954943,100555.149267,21.675226,Engerfjellet,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
2,2020-02-15 12:00:00,2020-02-15 14:00:00,2,3.116065,0.380970,0.847620,0.059136,276.095935,0.374416,197.767234,...,0.832413,100519.478200,44.229292,Engerfjellet,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
3,2020-02-15 12:00:00,2020-02-15 15:00:00,3,2.856627,0.388535,0.886108,0.062451,275.603674,0.339881,186.196795,...,0.792371,100468.608667,49.885909,Engerfjellet,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
4,2020-02-15 12:00:00,2020-02-15 16:00:00,4,3.303811,0.599790,0.942954,0.047610,274.942599,0.471217,185.264509,...,1.163573,100395.238133,66.165161,Engerfjellet,275.69696,100518.625,0.844108,0.0,2.051558,193.094101


In [135]:
merged.columns.to_list()

['time_ref',
 'time',
 'lt',
 'ws10m_mean',
 'ws10m_std',
 'rh2m_mean',
 'rh2m_std',
 't2m_mean',
 't2m_std',
 'wd10m_mean',
 'wd10m_std',
 'g10m_mean',
 'g10m_std',
 'mslp_mean',
 'mslp_std',
 'sid_clean',
 'air_temperature_2m',
 'air_pressure_at_sea_level',
 'relative_humidity_2m',
 'precipitation_amount',
 'wind_speed_10m',
 'wind_direction_10m']

In [136]:
len(merged), len(met_forecast)

(6874212, 6874212)

In [137]:
cols = merged.columns.tolist()

# Remove sid_clean from its current position
cols.remove("sid_clean")

# Insert it right after "time"
time_index = cols.index("time")
cols.insert(time_index + 1, "sid_clean")

# Reorder dataframe
merged = merged[cols]

merged.head()

,time_ref,time,sid_clean,lt,ws10m_mean,ws10m_std,rh2m_mean,rh2m_std,t2m_mean,t2m_std,...,g10m_mean,g10m_std,mslp_mean,mslp_std,air_temperature_2m,air_pressure_at_sea_level,relative_humidity_2m,precipitation_amount,wind_speed_10m,wind_direction_10m
0,2020-02-15 12:00:00,2020-02-15 12:00:00,Engerfjellet,0,2.139608,0.448171,0.823404,0.051589,276.379494,0.395486,...,6.050247,1.032772,100502.532600,38.879500,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
1,2020-02-15 12:00:00,2020-02-15 13:00:00,Engerfjellet,1,3.384011,0.476801,0.825629,0.047831,276.422947,0.375587,...,8.121981,0.954943,100555.149267,21.675226,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
2,2020-02-15 12:00:00,2020-02-15 14:00:00,Engerfjellet,2,3.116065,0.380970,0.847620,0.059136,276.095935,0.374416,...,8.276285,0.832413,100519.478200,44.229292,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
3,2020-02-15 12:00:00,2020-02-15 15:00:00,Engerfjellet,3,2.856627,0.388535,0.886108,0.062451,275.603674,0.339881,...,7.532237,0.792371,100468.608667,49.885909,275.69696,100518.625,0.844108,0.0,2.051558,193.094101
4,2020-02-15 12:00:00,2020-02-15 16:00:00,Engerfjellet,4,3.303811,0.599790,0.942954,0.047610,274.942599,0.471217,...,7.929002,1.163573,100395.238133,66.165161,275.69696,100518.625,0.844108,0.0,2.051558,193.094101


In [138]:
merged.to_parquet("../src/processed_data/notebook_data/met_forecast_nowcast_merged.parquet")